[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb)

# 🟢 Easy: Gradient Norm Clipping

Implement **gradient norm clipping** — a training stability technique.

### Signature
```python
def clip_grad_norm(parameters, max_norm: float) -> float:
    # Clip gradients in-place so total norm <= max_norm
    # Returns the original (unclipped) total norm
```

### Algorithm
1. Compute total norm: `sqrt(sum(p.grad.norm()^2 for p in parameters))`
2. If total > max_norm: scale all grads by `max_norm / total`
3. Return original total norm

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.1 MB/s eta 0:00:00


In [5]:
import torch
import math

In [15]:
# 本质其实很简单：统一缩放所有梯度（不是单独 是整体缩放）
# 如果梯度太大，就把整个梯度向量“缩短”，但方向不变
'''
常用默认值：
max_norm = 1.0
'''

def clip_grad_norm(parameters, max_norm):
    total_norm_sq = 0.0

    for p in parameters:
      if p.grad is None: continue
      param_norm = p.grad.norm(2)
      # p可能是一个张量
      total_norm_sq += param_norm.item() ** 2

    total_norm = math.sqrt(total_norm_sq)
    if total_norm > max_norm:
      scale = max_norm / (total_norm + 1e-6)
      for p in parameters:
        if p.grad is None: continue
        p.grad.mul_(scale) # 让新的total norm变成max norm

    return total_norm

In [16]:
# 🧪 Debug
p = torch.randn(100, requires_grad=True)
(p * 10).sum().backward()
print('Before:', p.grad.norm().item())
orig = clip_grad_norm([p], max_norm=1.0)
print('After: ', p.grad.norm().item())
print('Original norm:', orig)

Before: 100.0
After:  0.9999999403953552
Original norm: 100.0


In [17]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_clipping')


🧪 Testing: Gradient Norm Clipping (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Clips to max_norm (2.2ms)
  ✅ [2/4] Returns original norm (0.8ms)
  ✅ [3/4] No change when norm < max_norm (0.5ms)
  ✅ [4/4] Preserves direction (2.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (5.7ms total)
  Progress saved. Run status() to see your dashboard.

